In [36]:
import pandas as pd
# read the csv file
df = pd.read_csv("/eos/user/t/tcostaes/traccc_outputs/profiling/mldev02_roi_1t_1ev.csv")

In [37]:
# check total number of kernel launches
print(len(df))
event_size = len(df)//30 
print(event_size)

4320
144


In [38]:
df[df["Function Name"]=="ccl_kernel"].index

Index([   0,  144,  288,  432,  576,  720,  864, 1008, 1152, 1296, 1440, 1584,
       1728, 1872, 2016, 2160, 2304, 2448, 2592, 2736, 2880, 3024, 3168, 3312,
       3456, 3600, 3744, 3888, 4032, 4176],
      dtype='int64')

In [39]:
df["Function Name"].value_counts().head(70)

Function Name
DeviceRadixSortSingleTileKernel             870
apply_interaction                           540
find_tracks                                 540
fill_finding_propagation_sort_keys          510
propagate_to_next_surface                   510
fill_finding_duplicate_removal_sort_keys    360
remove_duplicates                           360
DeviceMergeSortMergeKernel                   60
static_kernel                                60
DeviceMergeSortPartitionKernel               60
count_grid_capacities                        30
populate_grid                                30
fill_sorted_measurements                     30
DeviceMergeSortBlockSortKernel               30
ccl_kernel                                   30
form_spacepoints                             30
count_doublets                               30
estimate_track_params                        30
select_seeds                                 30
update_triplet_weights                       30
find_triplets             

In [40]:
#split dataframe into events 
event_starts = df[df["Function Name"]=="ccl_kernel"].index.tolist()
events = []
for i, start in enumerate(event_starts):
    if i < len(event_starts)-1:
        event = df.iloc[start:event_starts[i+1]]
    else:
        event = df.iloc[start:]
    events.append(event)
print(len(events))

30


In [41]:
# calculate GPU time per event (5 cold run + 1 processed) * 5
event_times = []

for i, event in enumerate(events):
    total = event["Duration [ms]"].sum()
    event_times.append(total)
print(event_times)

[np.float64(19.5), np.float64(19.540000000000003), np.float64(19.55), np.float64(19.55), np.float64(19.48), np.float64(19.62), np.float64(19.59), np.float64(19.48), np.float64(19.54), np.float64(19.580000000000002), np.float64(19.66), np.float64(19.540000000000003), np.float64(19.5), np.float64(19.5), np.float64(19.57), np.float64(19.54), np.float64(19.610000000000003), np.float64(19.61), np.float64(19.57), np.float64(19.6), np.float64(19.560000000000002), np.float64(19.62), np.float64(19.599999999999998), np.float64(19.57), np.float64(19.6), np.float64(19.5), np.float64(19.53), np.float64(19.520000000000003), np.float64(19.52), np.float64(19.5)]


In [42]:
# take out cold runs
measured_events = events[5::6]
print(len(measured_events))

5


In [43]:
# average execution time 
measured_times = [
    event["Duration [ms]"].sum()
    for event in measured_events
]

print(measured_times)

average = sum(measured_times)/len(measured_times)

print("Average GPU time:", average, "ms")

[np.float64(19.62), np.float64(19.540000000000003), np.float64(19.61), np.float64(19.57), np.float64(19.5)]
Average GPU time: 19.568 ms


In [44]:
# find most expensive kernels 
measured_df = pd.concat(measured_events)
kernel_times = (
    measured_df.groupby("Function Name")["Duration [ms]"]
    .sum()
    / 5
)
kernel_times = kernel_times.sort_values(ascending=False)

print(kernel_times.head(20))

Function Name
propagate_to_next_surface          13.398
find_tracks                         1.510
find_doublets                       1.012
DeviceRadixSortSingleTileKernel     0.870
count_doublets                      0.802
DeviceMergeSortBlockSortKernel      0.714
ccl_kernel                          0.278
select_seeds                        0.238
remove_duplicates                   0.210
count_triplets                      0.134
find_triplets                       0.100
build_tracks                        0.080
DeviceMergeSortMergeKernel          0.070
DeviceMergeSortPartitionKernel      0.050
estimate_track_params               0.050
static_kernel                       0.010
populate_grid                       0.010
update_triplet_weights              0.010
form_spacepoints                    0.010
fill_sorted_measurements            0.010
Name: Duration [ms], dtype: float64
